In [1]:
from pathlib import Path
import onnxruntime as ort

ROOT = Path.cwd()
MODEL_DIR = ROOT / "models" / "onnx"

MODEL_FILES = [
    "segnet.onnx",
    "tromr_encoder.onnx",
    "tromr_decoder.onnx",
]

print("Project root:", ROOT)
print("Model dir:", MODEL_DIR)
print("ONNX Runtime:", ort.__version__)

Project root: d:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr
Model dir: d:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\models\onnx
ONNX Runtime: 1.26.0


In [2]:
available = ort.get_available_providers()

print("Available providers:")
for p in available:
    print(" -", p)

if "CUDAExecutionProvider" in available:
    print("\nGPU status: CUDA available")
else:
    print("\nGPU status: CUDA NOT available; ONNX will run on CPU only")

Available providers:
 - TensorrtExecutionProvider
 - CUDAExecutionProvider
 - CPUExecutionProvider

GPU status: CUDA available


In [3]:
cuda_options = {
    "device_id": 0,
    "cudnn_conv_algo_search": "HEURISTIC",
    "arena_extend_strategy": "kNextPowerOfTwo",
}

if "CUDAExecutionProvider" in ort.get_available_providers():
    providers = [
        ("CUDAExecutionProvider", cuda_options),
        "CPUExecutionProvider",
    ]
else:
    providers = ["CPUExecutionProvider"]

providers

[('CUDAExecutionProvider',
  {'device_id': 0,
   'cudnn_conv_algo_search': 'HEURISTIC',
   'arena_extend_strategy': 'kNextPowerOfTwo'}),
 'CPUExecutionProvider']

In [4]:
for name in MODEL_FILES:
    path = MODEL_DIR / name
    print(f"{name}: {path.exists()}")

    if not path.exists():
        print("  Missing:", path)

segnet.onnx: True
tromr_encoder.onnx: True
tromr_decoder.onnx: True


In [5]:
sessions = {}

for name in MODEL_FILES:
    path = MODEL_DIR / name

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    if not path.exists():
        print("SKIPPED: file does not exist")
        continue

    session = ort.InferenceSession(
        str(path),
        providers=providers,
    )

    sessions[name] = session

    print("Active providers:", session.get_providers())

    print("\nInputs:")
    for i in session.get_inputs():
        print(" ", i.name, i.shape, i.type)

    print("\nOutputs:")
    for o in session.get_outputs():
        print(" ", o.name, o.shape, o.type)


segnet.onnx
Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

Inputs:
  input ['batch_size', 3, 320, 320] tensor(float)

Outputs:
  output ['batch_size', 6, 320, 320] tensor(float)

tromr_encoder.onnx
Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

Inputs:
  input [1, 1, 256, 1280] tensor(float)

Outputs:
  output [1, 1280, 512] tensor(float)

tromr_decoder.onnx
Active providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

Inputs:
  rhythms [1, 1] tensor(int64)
  pitchs [1, 1] tensor(int64)
  lifts [1, 1] tensor(int64)
  articulations [1, 1] tensor(int64)
  context [1, 'cache_exists', 512] tensor(float)
  cache_len [1] tensor(int64)
  cache_in0 [1, 8, 'seq_len', 64] tensor(float)
  cache_in1 [1, 8, 'seq_len', 64] tensor(float)
  cache_in2 [1, 8, 'seq_len', 64] tensor(float)
  cache_in3 [1, 8, 'seq_len', 64] tensor(float)
  cache_in4 [1, 8, 'seq_len', 64] tensor(float)
  cache_in5 [1, 8, 'seq_len', 64] tensor(float)
  cache_in6 [1, 8, 'seq

In [6]:
required = set(MODEL_FILES)
loaded = set(sessions.keys())
missing = required - loaded

if not missing:
    print("All ONNX models loaded successfully.")
else:
    print("Missing or failed models:")
    for m in missing:
        print(" -", m)

All ONNX models loaded successfully.


In [7]:
cpu_sessions = {}

for name in MODEL_FILES:
    path = MODEL_DIR / name

    if not path.exists():
        continue

    cpu_sessions[name] = ort.InferenceSession(
        str(path),
        providers=["CPUExecutionProvider"],
    )

print("CPU-only sessions loaded:", list(cpu_sessions.keys()))

CPU-only sessions loaded: ['segnet.onnx', 'tromr_encoder.onnx', 'tromr_decoder.onnx']
